In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import pickle
import json 

# 2. Modèles et transformation de données
from sklearn.model_selection import train_test_split,GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error,mean_absolute_percentage_error
import statsmodels.api as sm
# 3. Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import datetime
# 2. FEATURE ENGINEERING — OLS -> filtre -> Lasso, par typologie
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LassoCV
from sklearn.compose import ColumnTransformer
import json


In [ ]:
def calc_best_alpha(alphas,X_origine,Y,n_split):
    """
    Cette fonction retourne le best_alpha calculé

    Arguments:
    alphas : Liste des alphas qui seront testés dans cette fonciton
    X_origine : X à utiliser pour le train_test_split
    Y : target
    """
    # Ajouter une constante ('const'), cette méthode est spécifique à statsmodels
    X = sm.add_constant(X_origine)

    X_train,X_test,Y_train,Y_test = train_test_split(X,Y,random_state=42,test_size=0.2) 

    # Initialiser la validation croisée
    kf = KFold(n_splits=n_split)
    best_alpha = None
    best_mse = float('inf')

    # Validation croisée pour trouver le meilleur alpha
    for alpha in alphas:
        mse_scores = []
        for train_index, test_index in kf.split(X):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            Y_train, Y_test = Y.iloc[train_index], Y.iloc[test_index]

            # Ajuster le modèle avec régularisation sqrt_lasso
            model = sm.OLS(Y_train, X_train).fit_regularized(method='sqrt_lasso', alpha=alpha)
            
            # Prédire sur l'ensemble de test
            Y_pred = model.predict(X_test)

            # Calculer l'erreur quadratique moyenne
            mse = mean_squared_error(Y_test, Y_pred)
            mse_scores.append(mse)

        # Calculer l'erreur quadratique moyenne
        mean_mse = np.mean(mse_scores)

        # Mettre à jour le meilleur alpha
        if mean_mse < best_mse:
            best_mse = mean_mse
            best_alpha = alpha

    # On retourne la meilleure valeur d'alpha calculée
    return best_alpha

In [ ]:
def select_target(target, df):
    """
    Cette fonciton permet de retourner le X et Y en fonction des paramètres 
    target : Nom de la target
    df : Dataframe sur le lequel on doit séparer la target (Y) des autres colonnes (X)
    """
    Y = df[target]
    X = df.drop(target, axis=1)
    # Si des données ne sont pas présentes, on remplit avec la valeur médiane
    X= X.fillna(X.median(numeric_only=True))

    return X,Y

In [ ]:
def Select_features_PValue(X_encoded,Y,target,new_rows):
    """
    Cette fonction permet de sélectionner les features que l'on garde en fonction de la pvalue calculer via le model OLS
    Par convention, on supprime les features ayant une pvalue supérieure à 0.05 (5%) une par une
    en partant de la pvalue la plusimportante à la moins importante
    Une fois cette sélection effectuée, on calcule les R² score
    On lance une dernière fois le modèle afin que les paramètres soit sur l'ensemble des lignes.
    """
    ok = True
    while ok:
        X_const=sm.add_constant(X_encoded)

        model = sm.OLS(Y,X_const)
        results = model.fit()

        results_df = pd.DataFrame({
            'coef': results.params,
            'std err': results.bse,
            't': results.tvalues,
            'P>|t|': results.pvalues
        })
        results_df_sorted = results_df[results_df['P>|t|'] > 0.05].sort_values(by="P>|t|", ascending=False)
        
        if len(results_df_sorted)>0 :
            feature_to_remove = results_df_sorted.index[0]
            if feature_to_remove != 'const':
                X_encoded = X_encoded.drop(feature_to_remove, axis=1)
            else:
                ok = False
        else:
            ok = False

    #Calcul des R²    
    X_train,X_test,Y_train, Y_test = train_test_split(X_const,Y,random_state=42, test_size=0.2)
   
    model = sm.OLS(Y_train,X_train).fit()

    Y_test_pred = model.predict(X_test)
    Y_train_pred = model.predict(X_train)

    new_rows[0]['R2 train'] = r2_score(Y_train,Y_train_pred)
    new_rows[0]['R2 test'] = r2_score(Y_test,Y_test_pred)

    rmse = np.sqrt(mean_squared_error(Y_test,Y_test_pred))
    new_rows[0]['RMSE'] = np.sqrt(mean_squared_error(Y_test,Y_test_pred)).round(0)
    new_rows[0]['MAE'] = mean_absolute_error(Y_test,Y_test_pred).round(0)
    new_rows[0]['MAPE'] = mean_absolute_percentage_error(Y_test,Y_test_pred).round(4)
    


    model = sm.OLS(Y,X_const).fit()
    with open(f'model_{target}.pkl','wb') as f:
        pickle.dump(model,f)

    return new_rows

In [ ]:
df_origine = pd.read_csv("../data/data_wip_v5.csv")
df = df_origine.drop(['Code_Dpt','année'],axis=1)
df.head()
df.info()

In [ ]:
df_results = pd.DataFrame(columns = 
    ['cible', 
    'Best_alpha',
    'R2 train',
    'R2 test',
    'RMSE',
    'MAE',
    'Valeur min',
    'Valeur max',
    'Moyenne',
    'Médiane']
    )

# targets = ['tonnage_dechet_produit','Total_autres_dechets','Déblais_gravats','Déchets_verts','Encombrants','Matériaux_recyclables']
targets = ['Déblais_gravats','Déchets_verts','Encombrants','Matériaux_recyclables']
# targets = ['Déblais_gravats']

nb_split = 6
nb_alphas = 20

# df_2019_envoi = df_origine[df_origine['année']==2019]
df_2019_envoi = df_origine
df_2019_dummies = pd.get_dummies(df_2019_envoi).astype(float)
df_2019_dummies.to_csv('df_dummies_total.csv',encoding='utf-8-sig')

for target in targets:
    print(target)
    new_rows = [
        {'cible': target, 
         'Valeur min':df[target].min(), 
         'Moyenne':df[target].mean().round(0), 
         'Valeur max':df[target].max(),
         'Médiane':df[target].median()
         }
    ]

    X,Y = select_target(target,df)
    
    X_encoded = pd.get_dummies(X).astype(float)

    alphas = np.logspace(-4, 3, nb_alphas)
    # print(alphas)

    #lancement du calcul des best alpha
    best_alpha = calc_best_alpha(alphas,X_encoded,Y,nb_split)
    
    alphas = np.arange(best_alpha*0.2,best_alpha*4,best_alpha*0.4)
    best_alpha = calc_best_alpha(alphas,X_encoded,Y,nb_split)
 
    new_rows[0]['Best_alpha'] = f"{best_alpha:.6f}"

    model = sm.OLS(Y,X_encoded).fit_regularized(method='sqrt_lasso', alpha=best_alpha)

    # Récupération des coefficients
    coefficients = model.params
        
    # Sélection des variables avec coef différent de zéro => qui ont un impact sur notre modèle
    selected_features = coefficients[coefficients != 0].index.tolist()
    
    #Nouvel X_encoded uniquement avec les colonnes sélectionnées
    X_encoded = X_encoded[selected_features]

    new_rows = Select_features_PValue(X_encoded,Y,target,new_rows)

    df_results = pd.concat([df_results, pd.DataFrame(new_rows)], ignore_index=True)

    print(f'Génération du modèle et des colonnes pour {target} est finie, {datetime.datetime.now()}')

print('Fin de la génération des modèles')
